# Day 28: Fine-Tuning and Domain Adaptation

A generic system is good at general language, but has no built-in reason to know your domain's own vocabulary. This notebook adapts a toy retriever and ranker to a small support knowledge base using labeled domain feedback, then measures the result on held-out data the adaptation never saw, the same idea behind real fine-tuning, at a scale that runs in seconds instead of hours on a GPU.

## The domain dataset and the train/test split

10 support documents, 25 labeled queries, split 60/40 into train and test before any tuning happens.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
from domain_dataset import DOCUMENTS, LABELED_QUERIES, split_queries
train_queries, test_queries = split_queries()
print('documents:', len(DOCUMENTS))
print('train queries:', len(train_queries))
print('test queries:', len(test_queries))


## Collect feedback

Simulated feedback from the train split only, with a small, seeded fraction of flipped labels standing in for real-world noise.

In [ ]:
import feedback_collector
feedback_collector.main()


## Adapt the vocabulary

Real domain terms (RMA, chargeback, backorder) found in train documents and train feedback, grouped as per-document concepts so retrieval can actually tell documents apart.

In [ ]:
import domain_vocabulary
domain_vocabulary.main()


## Tune ranker weights

A small grid search over similarity, recency, and quality weights, scored only on train feedback.

In [ ]:
import ranker_weight_tuning
ranker_weight_tuning.main()


## Held-out evaluation

The only numbers that actually count: Day 8's real precision, recall, NDCG, and MRR, measured on the test split neither the vocabulary nor the weights ever saw.

In [ ]:
import held_out_evaluation
held_out_evaluation.main()


## Overfitting check

The same adapted system, measured on train versus test. A gap where train looks better than test is the signature of overfitting.

In [ ]:
import overfitting_check
overfitting_check.main()


## Adaptation report

A short, honest summary: what changed, the real test metrics, which specific test queries improved or got worse, and a clear ship or do-not-ship recommendation.

In [ ]:
import adaptation_report
adaptation_report.main()


## Key takeaways

- A generic system has no built-in reason to know your domain's specific vocabulary, which is exactly what adaptation adds.
- The train/test split must happen before any tuning, splitting after the fact defeats the entire point.
- Real feedback is noisy, a small fraction of flipped labels is normal, not a sign something is broken.
- Tuning on train data will always look good on train data, only held-out test metrics tell you whether it actually generalizes.
- A held-out improvement can still hide a real regression on specific queries, which is why a per-query breakdown matters, not just an aggregate number.
- Real fine-tuning is expensive: labeling, compute, maintenance, and the risk of forgetting general knowledge. A measured gap should exist before paying that cost.